# 01 — Phase A: hosted-agent scenarios on the public lab

**Requirement.** Build and debug the hosted-agent scenarios (GA path, invoke, A2A, agent identity to a tool,
LangGraph, observability, cross-project) on a public lab, so that Phase B can replay them in the private
configuration that must be qualified.

**What this notebook does.** Read-only: deployment plan (dry-run), execution limits, pass/fail definitions and
recorded results per environment. It never deploys, never creates agents, never calls paid APIs and never cleans
up. Those actions run from the CLI after approval (`hapq lab deploy|run --phase A`, `hapq cleanup`), each dry-run
unless `--execute --confirm <resource-group>` is given.

## 1. Configuration and limits

In [ ]:
from IPython.display import Markdown, display

from hapq import report
from hapq.config import load_config, load_toml
from hapq.lab import deploy
from hapq.lab.profile import LabProfile
from hapq.ledger import Ledger
from hapq.redact import redact_text

cfg = load_config()
lim = cfg.limits
rows = [("Agent versions per run", lim.max_agent_versions), ("Sessions per run", lim.max_sessions),
        ("Calls per run", lim.max_calls), ("Tokens per run", lim.max_total_tokens),
        ("Output tokens per call", lim.max_output_tokens), ("Concurrency", lim.max_concurrency),
        ("Sandbox", f"{lim.cpu} vCPU / {lim.memory}, idle timeout {lim.idle_timeout_s}s"),
        ("Timeouts", f"call {lim.call_timeout_s}s, deploy {lim.deploy_timeout_s}s, traces {lim.trace_wait_s}s")]
display(Markdown("| Limit (client side, NOT a billing cap) | Value |\n|---|---|\n" + "\n".join(f"| {k} | {v} |" for k, v in rows)))

## 2. Deployment plan (dry-run, nothing is created)

In [ ]:
steps = deploy.plan_phase_a(cfg, suffix="lab01", budget_emails=["<recipient>"])
display(Markdown("| # | Step | What | Creates |\n|---|---|---|---|\n" + "\n".join(
    f"| {i} | {s.name} | {s.description} | {s.creates} |" for i, s in enumerate(steps, 1))))
try:
    LabProfile.load("A")
    print("lab A profile present: lab A has been deployed by this kit")
except FileNotFoundError:
    print("lab A not deployed (no profile)")
print(f"objects owned by the kit in A (ledger): {len(Ledger(cfg.ledger_path).owned('A'))}")

## 3. Pass/fail definitions of the Phase A scenarios

In [ ]:
scen = [s for s in load_toml("criteria.toml")["scenario"] if any(r.startswith("A@") for r in s["runs"])]
display(Markdown("| Scenario | Implemented for A | PASS if | FAIL if |\n|---|---|---|---|\n" + "\n".join(
    f"| {s['id']} | {'yes' if any(r.startswith('A@') for r in s.get('implemented', [])) else 'no'} | {s['pass_if']} | {s['fail_if']} |"
    for s in scen)))

## 4. Results: documentary status and execution verdicts, kept apart

Each environment's verdict uses only its own execution results. Network criteria (C02.*) are judged on B only.
Criteria with partial coverage cannot exceed PARTIAL.

In [ ]:
table, _ = report.build_matrix(cfg)
out = ["| Criterion | Documentary | Verdict A | Verdict B (target) | Coverage limit |", "|---|---|---|---|---|"]
for c in table:
    doc = ", ".join(sorted({("contradiction" if r.details.get("contradiction_present") else r.status) for r in c["docs"]})) or "-"
    out.append(f"| {c['id']} | {doc} | {c['verdict_A']} | {c['verdict_B']} | {c['coverage_limit'] or '-'} |")
display(Markdown(redact_text("\n".join(out))))

## 5. Recorded Azure evidence (environment A, westeurope)

Read from `results/` (redacted JSONL written by the CLI at execution time). Nothing below is re-executed:
each row shows the run id and UTC time of the recorded execution.

In [ ]:
latest = report.latest_results(cfg)
show = ["S01-GA-CREATE", "S09-OBS-STATUS", "S02-INVOKE", "S10-OBS-LOGS", "S11-OBS-TRACES", "S08-LANGGRAPH",
        "S03-A2A-CARD", "S05-A2A-NO-TOKEN-401", "S04-A2A-MESSAGE", "S06-A2A-NO-ROLE-403",
        "S14-MCP-NO-TOKEN-401", "S15-MCP-WRONG-AUDIENCE-401", "S07-TOOL-AGENT-IDENTITY"]
rows = ["| Scenario | Status | Recorded (UTC) | Run | Observation |", "|---|---|---|---|---|"]
for sid in show:
    for (cid, _, env, _pt), r in latest.items():
        if cid == sid and env == "A":
            rows.append(f"| {sid} | {r.status} | {r.timestamp[:19]} | {r.run_id} | {r.observed[:150]} |")
display(Markdown(redact_text("\n".join(rows))))
extras = {sid: r.details for (sid, _, env, _), r in latest.items() if env == "A"}
s02 = extras.get("S02-INVOKE", {}).get("calls", [])
lg = extras.get("S08-LANGGRAPH", {})
display(Markdown(redact_text("\n".join(
    ["**Recorded answers (synthetic prompts):**"]
    + [f"- S02 call {i + 1}: `{c.get('answer')}` (usage {c.get('usage')})" for i, c in enumerate(s02)]
    + [f"- S08 LangGraph: `{lg.get('answer')}`",
       f"- Versions installed by the platform for the LangGraph agent (container log): `{lg.get('platform_installed_versions')}`",
       f"- S07 token seen by the MCP tool: idtyp `{extras.get('S07-TOOL-AGENT-IDENTITY', {}).get('idtyp')}`, "
       f"claims `{extras.get('S07-TOOL-AGENT-IDENTITY', {}).get('claim_names')}` (values not shown)"]))))

## 6. Live check (read-only, opt-in)

Disabled by default so that "Run All" only reads recorded evidence. Set the environment variable
`HAPQ_NOTEBOOK_LIVE=1` before starting the kernel to read, live: the status of the lab agents and the A2A agent card.
No agent is created, no model is called, nothing is modified.

In [ ]:
import os

RUN_LIVE = os.environ.get("HAPQ_NOTEBOOK_LIVE") == "1"
if not RUN_LIVE:
    print("live check skipped (HAPQ_NOTEBOOK_LIVE != 1): the tables above are recorded evidence only")
else:
    import httpx
    from azure.ai.projects import AIProjectClient
    from azure.identity import AzureCliCredential
    from datetime import datetime, timezone

    profile = LabProfile.load("A")
    cred = AzureCliCredential(process_timeout=60)
    client = AIProjectClient(endpoint=profile.project_endpoints[0], credential=cred, read_timeout=60, retry_total=1)
    live = ["| Agent | Latest version | Status (live) |", "|---|---|---|"]
    for name in ("hapq-min", "hapq-lg", "hapq-a2a", "hapq-mcp"):
        d = client.agents.get(agent_name=name).as_dict()
        v = ((d.get("versions") or {}).get("latest") or {})
        live.append(f"| {name} | {v.get('version')} | {v.get('status')} |")
    token = cred.get_token("https://ai.azure.com/.default").token
    card = httpx.get(f"{profile.project_endpoints[0]}/agents/hapq-a2a/endpoint/protocols/a2a/agentCard/v1.0",
                     headers={"Authorization": f"Bearer {token}"}, timeout=60)
    print("live read at", datetime.now(timezone.utc).isoformat(timespec="seconds"))
    tool = httpx.get(f"{profile.mcp_url}/healthz", timeout=60).status_code if profile.mcp_url else "not deployed"
    display(Markdown(redact_text("\n".join(live) + f"\n\nA2A agent card (live): HTTP {card.status_code}"
                                 + f"\n\nMCP tool /healthz (live): HTTP {tool}")))

## 7. Limits

- Phase A is public: it cannot qualify a network criterion, and an A success never qualifies B.
- S01 shows creation works on the GA API without preview enrollment; the GA status itself stays documentary.
- S08: LangGraph works with the agent identity through the Responses API; the chat-completions path of the Learn sample
  was refused (401) in this lab. Observation, not a documented rule.
- S03-S05: incoming A2A can be enabled and the card is served for a hosted agent, but SendMessage is refused by the
  platform (HOSTED_AGENT_NOT_SUPPORTED) as of 2026-10-02.
- S07 proves the agent's own (instance) identity reaches a tool through the toolbox (token validated by the tool, oid =
  instance identity, idtyp=app). It does not cover end-user propagation (OBO / OAuth passthrough).
- S07 uses agent-framework-foundry-hosting 1.0.0b260918 with the package's own single-instance pattern; the Learn
  per-request factory pattern was not accepted by any of the 5 published versions examined (documented divergence).
- S06: an identity without Foundry role gets 404 (resource hidden) where the same request with a role gets 200.
- S12/S13 cover cross-project consumption and inventory inside one subscription; no enterprise registry.
- Kit limits bound the kit's own activity; they are not a billing cap. Budget alerts notify, they do not stop spend.